# RFP 전처리 실험 및 개선 이력

실제 변경 원장과 원본 재실행 결과 비교. 전체 과정 Notebook과 별도 실행 가능.

실행 출력: Notebook별 독립 IPython namespace에서 셀 순서 검증. Jupyter 커널 실행 검증은 별도.

### 공개용 경로 정리 및 실행 범위 (2026-10-19)

저장된 표와 통계는 과거 실행 결과입니다. 개인정보 경로와 서버 주소만 마스킹했으며 이번 공개 정리에서 전체 노트북을 재실행하지 않았습니다. 저장된 PASS는 과거 실행의 판정이며 현재 환경의 실행 검증을 뜻하지 않습니다. 원문 보존 검증은 파싱 정확도·검색 성능·답변 정확도와 별개입니다.

저장소 루트는 `pyproject.toml`, `rag/`, `preprocessing/`로 식별합니다. 외부 자료는 저장소에 포함되지 않습니다. `RFP_EXTERNAL_ROOT`를 실제 외부 자료 번들의 루트로 설정하고 다음 하위 트리를 제공해야 합니다: `project/`(기존 전처리 작업 폴더, `preprocessing/settings.json`, `run_state.json`, `original_sources/legacy`, Native 자료와 원문 데이터), `history/`(날짜별 과거 작업 폴더와 검증 원장), `archives/`(최종 패키지와 이전 저장소 백업), `packages/`(이전 검증 패키지). 이 트리는 자동 생성하지 않습니다. 각 환경의 실제 파일에 맞게 로컬 설정 JSON의 경로도 수정해야 합니다. 원문 HWP/PDF와 검증 DB·Stage별 산출물이 없으면 관련 셀은 실행 미검증입니다.

외부 설정 파일은 `RFP_SETTINGS_FILE`, 실행 상태 파일은 `RFP_RUN_STATE_FILE`로 별도 지정할 수 있습니다. 기본 위치는 외부 `project/preprocessing/`입니다. Python과 HWP 변환기는 `RFP_LEGACY_PYTHON`, `RFP_HWP5PROC`로 실제 실행 파일을 설정해야 합니다. `settings.json`에는 기존 `project`, `historical_root`, `final_package`, `legacy_python`, `chunk_python`, `native_dataset` 경로가 필요합니다. `run_state.json`에는 기존 단계별 산출물 경로가 필요합니다. 개인 경로가 있는 설정 파일을 커밋하지 마세요.

`${RFP_EXTERNAL_ROOT}` 표기는 과거 출력 및 내장 코드의 비공개 경로 치환입니다. 과거 코드의 기록된 SHA-256은 마스킹 이전 원본의 해시로 유지합니다. 역사 코드 재실행은 외부 원본 파일을 그 해시로 검증한 뒤 사용하며 기본값은 비활성화입니다. 내장 코드의 경로가 마스킹되어 있으므로 원본 바이트와 같다고 해석하지 마세요. 새 출력은 저장소의 `work/`에만 생성합니다.


## 1. 경로와 의존성

`BASE`는 기존 9개 Notebook 폴더. 경로를 옮기면 이 셀에서 변경.

- 외부 코드: `original_sources/legacy/src`의 파싱·정제·복구 함수.
- 전체 실행: `replay_initial.py`, `replay_mapping.py`, 최종 패키지의 `adapter/reproduce_elements.py`.
- 외부 데이터: `PROJECT` 원본·검토 기록, `HISTORY` 단계별 결과, `FINAL` 최종 패키지.
- 실행 환경: 기존 `rfp-rag` Python, pandas, IPython, HWP loader, PyMuPDF, pyhwp/hwp5proc. Mapper는 기존 별도 Python과 패키지·토큰 캐시 사용.

경로 설정·출력·비교 함수는 아래 셀에 포함. 데이터 확인 셀은 외부 Python 실행 없이 동작하지만 기존 데이터 파일이 필요하다. 파싱·정제·Elements 생성은 외부 코드가 필요하다.

Mapper 내부의 토큰 수·표 행 유틸리티 의존성은 유지. 청크 생성·분할 함수는 실행하지 않는다.

In [1]:
import os
from pathlib import Path

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "rag").is_dir() and (candidate / "preprocessing").is_dir():
            return candidate
    raise FileNotFoundError("rfg_rag 저장소 또는 그 하위 폴더에서 실행하세요.")

PROJECT_ROOT = find_project_root()
if not os.environ.get("RFP_EXTERNAL_ROOT"):
    raise RuntimeError("외부 역사 자료 번들의 루트를 RFP_EXTERNAL_ROOT로 설정하세요. 첫 Markdown의 설정 안내를 확인하세요.")
from pathlib import Path
from collections import Counter, defaultdict
from itertools import zip_longest
from IPython.display import display
import sys, json, re, copy, os, hashlib, subprocess, uuid, unicodedata
import pandas as pd

# 원본 폴더는 읽기 전용 입력. 새 결과는 현재 작업의 work에 저장한다.
BASE = Path(os.path.expandvars('${RFP_EXTERNAL_ROOT}/project/preprocessing'))
WORK = PROJECT_ROOT / 'work' / 'preprocessing_experiments'
SETTINGS = json.loads(Path(os.environ.get('RFP_SETTINGS_FILE', str(BASE / 'settings.json'))).read_text())
PROJECT = Path(SETTINGS['project'])
HISTORY = Path(SETTINGS['historical_root'])
FINAL = Path(SETTINGS['final_package'])
LEGACY = BASE / 'original_sources/legacy'
sys.dont_write_bytecode = True
sys.path.insert(0, str(LEGACY / 'src'))
RUN_STATE = json.loads(Path(os.environ.get('RFP_RUN_STATE_FILE', str(BASE / 'run_state.json'))).read_text())
# 전체 과정 Notebook의 새 실행 결과가 있으면 같은 입력으로 비교한다.
previous_runs = sorted((WORK / 'preprocessing_runs').glob('validation.json_*'), key=lambda p: p.stat().st_mtime)
if previous_runs:
    RUN_STATE.update(json.loads(previous_runs[-1].read_text())['run_paths'])

def read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def jl(path):
    with Path(path).open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def clip(value, n=650):
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, default=str)
    return text if len(text) <= n else text[:n] + ' …'

def frame(rows):
    with pd.option_context('display.max_colwidth', 120, 'display.max_rows', 12, 'display.max_columns', 12):
        display(pd.DataFrame(rows))

def state():
    return RUN_STATE

def save_state(values):
    RUN_STATE.update({k: str(v) for k, v in values.items()})

def unique_output(label):
    path = WORK / 'preprocessing_runs' / f'{label}_{uuid.uuid4().hex[:10]}'
    path.parent.mkdir(parents=True, exist_ok=True)
    assert not path.exists()
    return path

def command(python, script, args, label):
    log = unique_output(label + '.log')
    env = os.environ.copy()
    env['PYTHONDONTWRITEBYTECODE'] = '1'
    with log.open('w') as f:
        result = subprocess.run([str(python), str(script), *map(str, args)], stdout=f, stderr=subprocess.STDOUT, env=env)
    if result.returncode:
        raise RuntimeError(f'실행 실패({result.returncode}): {log}')
    print(log.read_text()[-1800:])
    return log

def data_versions():
    data = next(p for p in PROJECT.iterdir() if unicodedata.normalize('NFC', p.name) == '원본 데이터')
    return data / 'RAG_정제데이터'

def checks_equal_jsonl(a, b):
    # 순서·본문·메타데이터를 포함한 전체 행 비교.
    count = 0
    for count, (x, y) in enumerate(zip_longest(jl(a), jl(b)), 1):
        assert x == y, f'행 {count} 불일치'
    return {'행 수': count, '전체 값 동일': True, '바이트 해시 동일': sha(a) == sha(b)}

print('원본:', PROJECT)
print('새 실행 결과:', WORK / 'preprocessing_runs')

원본: ${RFP_EXTERNAL_ROOT}/project
새 실행 결과: ${RFP_EXTERNAL_ROOT}/history/2026-10-12/notebook-9-notebook-2-1-01-2/work/preprocessing_runs


## 2. 문자 깨짐 전후

탭 부가정보 오독과 승인된 PDF 글리프 복구. 정상 한자·기호는 일괄 삭제하지 않음.

In [2]:
from preprocess import inventory, process, SAMPLE
original_root, documents = inventory(PROJECT)
row = next(r for r in documents if r['doc_id']==SAMPLE)
initial = Path(state()['initial'])
raw = read(initial/'raw'/f"{row['doc_id']}.json")
# 실제 process 호출. 학습 출력만 추가하고 원본 함수는 변경하지 않습니다.
units, changes, exceptions, links, report = process(row,raw,PROJECT/'artifacts/preprocessing')
changed = [u for u in units if u['raw_text']!=u['clean_text'] or u['clean_text']!=u['search_text']]
frame([{'record_id':u['record_id'],'정제 전':clip(u['raw_text']),'정제 후':clip(u['clean_text']),'검색 표현':clip(u['search_text']),'요구사항 ID':u['requirement_ids']} for u in changed[:3]])
frame([{'규칙':c.get('rule',c.get('operation')),'좌표 공간':c.get('coordinate_space'),'전':clip(c.get('original',c.get('before'))),'후':clip(c.get('replacement',c.get('after')))} for c in changes[:5]])

,record_id,정제 전,정제 후,검색 표현,요구사항 ID
0,3c832bf48d6778590797a18328a78282662cd111d6f3b0b4b06362b4fc514a18:body:0:0-100,"| | |\n| --- | --- |\n| 2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 |\n| | |\n\n","| --- | --- |\n| 2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 |\n\n","| --- | --- |\n| 2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 |\n\n",[]
1,3c832bf48d6778590797a18328a78282662cd111d6f3b0b4b06362b4fc514a18:body:0:119-168,| | 목 차 | |\n| --- | --- | --- |\n| | |\n| |\n\n,,,[]
2,3c832bf48d6778590797a18328a78282662cd111d6f3b0b4b06362b4fc514a18:body:0:168-295,목 차\n 1. 추진개요\t螨ȃ\t 3\n 2. 추진방안\t螨ȃ\t 5\n 3. 추진내용\t螨ȃ\t 9\n 4. 제안요청내용\t砼ȃ\t 24\n 5. 입찰관련사항\t砼ȃ\t 78\n 6. 제안서작성요령\t牠...,,,[]


,규칙,좌표 공간,전,후
0,모든 셀이 빈 표 행 제거,승인 변경 후 구간,| | |\n,
1,모든 셀이 빈 표 행 제거,승인 변경 후 구간,| | |\n,
2,승인된 앞쪽 목차 제외,원문 구간,| | 목 차 | |\n| --- | --- | --- |\n| | |\n| |\n\n,
3,승인된 앞쪽 목차 제외,원문 구간,목 차\n 1. 추진개요\t螨ȃ\t 3\n 2. 추진방안\t螨ȃ\t 5\n 3. 추진내용\t螨ȃ\t 9\n 4. 제안요청내용\t砼ȃ\t 24\n 5. 입찰관련사항\t砼ȃ\t 78\n 6. 제안서작성요령\t牠...,
4,검색 표현에 한자 독음 병기,정제문 구간,本,本(본)


In [3]:
repaired = Path(state()['repaired'])
repair_changes = list(jl(repaired/'문자복구_변경내역.jsonl'))
frame([{k:clip(v,500) for k,v in r.items() if k in ['record_id','before','after','rule','checks','원본','복구']} for r in repair_changes[:3]])
print('이번 전체 복구 실행 변경 기록:',len(repair_changes))
# 실제 JSON 구조도 1개 확인합니다. 모든 값 출력 대신 key와 대표 값만 봅니다.
frame([{'필드':k,'실제 값':clip(v,500)} for k,v in repair_changes[0].items()][:12])
recheck_dir=Path(state()['rechecked'])
frame([read(recheck_dir/'예외_재검사_검증.json')])

,record_id,before,after,checks,rule
0,0205e3319d99d17c7aff6808007ef34bfa5484d32b24a826ecda94de9eaabc52:body:0:109-187,Ⅰ. 제안 요청내용\t矰ȃ\t 01\n 1. 용역개요\t脨ȃ\t 02 \n 2. 기본방향\t脨ȃ\t 03\n 3. 세부 과업지침\t獼ȃ\t 05 \n,Ⅰ. 제안 요청내용\t\t 01\n 1. 용역개요\t\t 02 \n 2. 기본방향\t\t 03\n 3. 세부 과업지침\t\t 05 \n,"{""수치_검사"": true, ""추출_숫자열_변경전"": [""01"", ""1"", ""02"", ""2"", ""03"", ""3"", ""05""], ""원문_수치열_변경전"": [""01"", ""1"", ""02"", ""2"", ""03"", ""3...",원본 근거 문자 복구
1,0205e3319d99d17c7aff6808007ef34bfa5484d32b24a826ecda94de9eaabc52:body:0:187-403,Ⅲ. 입찰 안내사항\t瞸ȃ\t 13\n 1. 입찰참가자격\t瘸ȃ\t 13\n 2. 입찰 및 낙찰자 선정\t挔ȃ\t 14\n 3. 입찰참가 등록 및 입찰서 제출 \t䞼ȃ\t 15\n 4. 입찰시 ...,Ⅲ. 입찰 안내사항\t\t 13\n 1. 입찰참가자격\t\t 13\n 2. 입찰 및 낙찰자 선정\t\t 14\n 3. 입찰참가 등록 및 입찰서 제출 \t\t 15\n 4. 입찰시 유의사항 \t\...,"{""수치_검사"": true, ""추출_숫자열_변경전"": [""13"", ""1"", ""13"", ""2"", ""14"", ""3"", ""15"", ""4"", ""16"", ""5"", ""17"", ""6"", ""17"", ""7"", ""22"", ""8...",원본 근거 문자 복구
2,0205e3319d99d17c7aff6808007ef34bfa5484d32b24a826ecda94de9eaabc52:body:0:403-463,Ⅳ. 제안서 작성 안내사항\t撔ȃ\t 25\n 1. 제안 안내사항\t瘸ȃ\t25\n 2. 제안서 작성방법\t27\n,Ⅳ. 제안서 작성 안내사항\t\t 25\n 1. 제안 안내사항\t\t25\n 2. 제안서 작성방법\t27\n,"{""수치_검사"": true, ""추출_숫자열_변경전"": [""25"", ""1"", ""25"", ""2"", ""27""], ""원문_수치열_변경전"": [""25"", ""1"", ""25"", ""2"", ""27""], ""수치열_변경후"": [...",원본 근거 문자 복구


이번 전체 복구 실행 변경 기록: 391


,필드,실제 값
0,doc_id,0205e3319d99d17c7aff6808007ef34bfa5484d32b24a826ecda94de9eaabc52
1,record_id,0205e3319d99d17c7aff6808007ef34bfa5484d32b24a826ecda94de9eaabc52:body:0:109-187
2,original_exception_id,검토-00003
3,coordinate_space,이전 버전 정제문
4,before,Ⅰ. 제안 요청내용\t矰ȃ\t 01\n 1. 용역개요\t脨ȃ\t 02 \n 2. 기본방향\t脨ȃ\t 03\n 3. 세부 과업지침\t獼ȃ\t 05 \n
5,after,Ⅰ. 제안 요청내용\t\t 01\n 1. 용역개요\t\t 02 \n 2. 기본방향\t\t 03\n 3. 세부 과업지침\t\t 05 \n
6,search_before,Ⅰ. 제안 요청내용 矰ȃ 01\n 1. 용역개요 脨ȃ 02 \n 2. 기본방향 脨ȃ 03\n 3. 세부 과업지침 獼ȃ 05 \n
7,search_after,Ⅰ. 제안 요청내용 01\n 1. 용역개요 02 \n 2. 기본방향 03\n 3. 세부 과업지침 05 \n
8,edits,"[{""start"": 11, ""end"": 13, ""original"": ""矰ȃ"", ""replacement"": """", ""rule"": ""원본 탭 제어 부가정보의 문자 오독 복구"", ""evidence_token"": ""..."
9,checks,"{""수치_검사"": true, ""추출_숫자열_변경전"": [""01"", ""1"", ""02"", ""2"", ""03"", ""3"", ""05""], ""원문_수치열_변경전"": [""01"", ""1"", ""02"", ""2"", ""03"", ""3..."


,상태,재검사_이전예외,현재_남은예외,원문좌표_유지,정제문좌표_현재문자열_대조,본문_수치_ID_조건_무변경,원본_완전성_미확인,구조항목_임의해결,복구근거_소멸확인
0,검사 통과,6115,5529,True,True,True,True,False,588


### 2. 문자 정제 및 깨짐 복구 전후 비교

- `process()` 함수를 실행하여 원본 텍스트와 정제 후 텍스트, 검색 표현을 비교했다.
- 표 행의 불필요한 구분 기호와 승인된 앞쪽 목차를 제거하고, 검색 표현의 한자 독음을 병기한 사례를 확인했다.
- 원본 근거를 바탕으로 문자 깨짐 복구 전후의 텍스트를 비교했다.
- 이번 복구 실행에서 총 391건의 변경 기록이 확인됐다.
- 예외 재검사 결과, 기존 6,115건 중 5,529건이 남아 있으며 복구 근거가 확인된 항목은 588건이다.
- 원문 좌표 유지, 정제 후 텍스트의 현재 문자 위치 대응, 본문·수치 ID 조건 검사는 모두 `True`로 확인됐다.
- 구조 항목의 임의 해결 여부는 `False`로 나타났다.
- 일부 예외가 남아 있어 문자 복구가 모두 완료된 상태는 아니다.

## 3. 표 구조 보존 실험

평탄화로 중복된 문구와 끊어진 행 연결 확인. 원본 row/column·rowspan/colspan으로 재직렬화.

In [4]:
from preprocess import inventory
from structure_source import HWP5Reader, table_grid
from repair_confirmed_tables import 원본직렬화, 표식보존
original_root, documents=inventory(PROJECT)
bydoc={r['doc_id']:r for r in documents}
ledger_path=data_versions()/'version_04_tables_20261007T021326_76e75bc3/표복구_변경내역.json'
ledger=read(ledger_path)
actual=[];grids=[]
for entry in ledger:
    # 실제 과거 변경 후 구조의 원본 주소를 사용합니다. 새 주소를 추측하지 않습니다.
    structure=entry['변경']['table_structure']['후']
    path=original_root/bydoc[entry['doc_id']]['representative_path']
    assert sha(path)==entry['doc_id']
    reader=HWP5Reader(path)
    records=list(reader._parse_records(reader._read_section(structure['section'])))
    grid=table_grid(records,structure['root_table_record'],reader)
    assert grid is not None and grid['source_table_sha256']==structure['source_table_sha256']
    rendered,new_structure,relations=원본직렬화(reader,records,grid)
    reader.close()
    before=entry['변경']['clean_text']['전'];expected=entry['변경']['clean_text']['후']
    rendered=표식보존(rendered,before)
    assert rendered==expected, '과거 복구 후 텍스트와 재실행 결과 불일치'
    assert len(new_structure['consumed_paragraph_records'])==len(set(new_structure['consumed_paragraph_records']))
    actual.append({'record_id':entry['record_id'],'개선 전':clip(before,1400),'원본 재직렬화 후':clip(rendered,1400),'과거 결과 동일':True,'하위 표 관계':len(relations)})
    grids.extend([{'문서':entry['doc_id'][:12], 'table_record':t['source_table_record'],'row':c['row'],'column':c['column'],'rowspan':c['rowspan'],'colspan':c['colspan'],'셀 원문':clip(c['text'],250)} for t in new_structure['tables'] for c in t['cells']])
frame(actual)
frame(grids[:10])
print('원본 재직렬화 재현 성공 위치:',len(actual))

,record_id,개선 전,원본 재직렬화 후,과거 결과 동일,하위 표 관계
0,699085551ebca10d106afc3234ce1fa1292be6196727e5ace2c9331c5f4c59b4:body:0:3205-3553,"| [IMAGE] 비전 비전 [IMAGE] 미래지식 교육을 선도하는 서울교육 미래지식 교육을 선도하는 서울교육 * DNA: DATA, Network, AI(인공지능) 전략 목표 전략 목표 전략 과제 전략 과제...",| [IMAGE] 비전 | [IMAGE] 미래지식 교육을 선도하는 서울교육 |\n| --- | --- |\n| 목표 | • AI·SW·정보보호 교육 확대를 통한 미래인재육성 • 미래 정보인프라 구축으로 디지털...,True,4
1,7e07a7f9ecf4fea73d89ab882d7e43624844e54dc595213676ba1b4e9ad5d76f:body:0:149-430,| | 〈청렴한 업무처리 및 부조리 신고〉 | |\n| --- | --- | --- |\n| ‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’ ‘계약과정 및 계약이행 시 금품·향응·편...,| | 〈청렴한 업무처리 및 부조리 신고〉 | |\n| --- | --- | --- |\n| | | |\n| ‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’ ‘계약과정 및 계약...,True,1
2,9e7d457cd7643ef6db50430a0a5c3ef6bfaba5d43f3b68e5e0467366a9067ede:body:0:96-380,| | 〈청렴한 업무처리 및 부조리 신고 안내〉 | |\n| --- | --- | --- |\n| ‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’ ‘계약과정 및 계약이행 시 금품·향...,| | 〈청렴한 업무처리 및 부조리 신고 안내〉 | |\n| --- | --- | --- |\n| | | |\n| ‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’ ‘계약과정 및...,True,1


,문서,table_record,row,column,rowspan,colspan,셀 원문
0,699085551ebc,649,0,0,1,1,비전 미래지식 교육을 선도하는 서울교육 목표 • AI·SW·정보보호 교육 확대를 통한 미래인재육성 • 미래 정보인프라 구축으로 디지털 서울교육 견인 • DNA 기반으로 특화교육을 선도하는 서울교육 서비스 제공...
1,699085551ebc,656,0,0,1,1,비전
2,699085551ebc,656,0,1,1,1,미래지식 교육을 선도하는 서울교육
3,699085551ebc,656,1,0,1,1,목표
4,699085551ebc,656,1,1,1,1,• AI·SW·정보보호 교육 확대를 통한 미래인재육성 • 미래 정보인프라 구축으로 디지털 서울교육 견인 • DNA 기반으로 특화교육을 선도하는 서울교육 서비스 제공
5,699085551ebc,689,0,0,1,1,미래지식 교육을 선도하는 서울교육
6,699085551ebc,719,0,0,1,1,• AI·SW·정보보호 교육 확대를 통한 미래인재육성 • 미래 정보인프라 구축으로 디지털 서울교육 견인 • DNA 기반으로 특화교육을 선도하는 서울교육 서비스 제공
7,699085551ebc,742,0,0,1,1,전략 목표
8,699085551ebc,742,0,1,1,1,전략 과제
9,699085551ebc,742,1,0,1,1,AI·SW·정보보호 교육 확대를 통한 미래인재육성


원본 재직렬화 재현 성공 위치: 3


### 3. 표 구조 보존 실험

- 표 구조 개선 전후의 텍스트를 비교했다.
- 원본 표의 행(row), 열(column), 병합 정보(rowspan, colspan)를 확인했다.
- 표 셀의 원문과 좌표 정보를 유지하면서 텍스트가 재직렬화된 결과를 확인했다.
- 과거 결 과 동일 여부는 비교한 3개 Record 모두 `True`로 나타났다.
- 원본 재직렬화 결과에서 차이가 발생한 위치는 3곳으로 확인됐다.

## 4. Native 파서 대조

기존 격리 방식으로 운영 상대 import를 제외하고 대표 HWP/PDF 재파싱. 전체 Native 재추출은 수행하지 않음.

In [5]:
import ast, types, hashlib
native_path=BASE/'original_sources/native_ingestion_source.py'
tree=ast.parse(native_path.read_text())
# 기존 격리 runner와 동일하게 운영 저장소 상대 import만 제외합니다.
tree.body=[node for node in tree.body if not isinstance(node,ast.ImportFrom) or node.level==0]
native={'__name__':'learning_isolated_native'}
exec(compile(tree,str(native_path),'exec'),native)
native_status=read(HISTORY/'work/stage8_2QA/native/parse_status.json')
native_inputs=read(HISTORY/'work/stage8_2QA/native/inputs.json')
source_by_hash={r['source_hash']:r for r in native_inputs}
chosen=[next(r for r in native_status if r['route']=='native' and not r['reason'] and source_by_hash[r['source_hash']]['format']==kind) for kind in ['hwp','pdf']]
xml_temp=unique_output('02_native_xml');xml_temp.mkdir()
settings=types.SimpleNamespace(data_dir=xml_temp,hwp_converter=Path(os.path.expandvars('${RFP_HWP5PROC}')),converter_timeout_seconds=300)
native_samples=[];native_summary=[]
for status in chosen:
    original=source_by_hash[status['source_hash']]
    path=Path(original['local_original'])
    assert sha(path)==status['source_hash']
    if original['format']=='hwp':
        parsed,warnings,reason=native['parse_hwp'](settings,path)
    else:
        parsed,warnings,reason=native['parse_pdf'](path)
    assert reason is None,reason
    finalized=native['finalize_elements'](parsed,status['extraction_id'])
    for element in finalized:element['extraction_id']=status['extraction_id']
    native_samples.extend(finalized[:2])
    native_summary.append({'형식':original['format'],'실제 Native 재추출 elements':len(finalized),'과거 elements':status['elements'],'수 동일':len(finalized)==status['elements'],'warnings':clip(warnings,500),'reason':reason})
frame(native_summary)
frame([{'kind':e['kind'],'원문':clip(e['raw_text']),'원본 location':clip(e['location']),'격자':clip(e.get('table'))} for e in native_samples])
frame([{'과거 Native 상태':kind,'문서 수':count} for kind,count in Counter('실패: '+str(r['reason']) if r['reason'] else '성공' for r in native_status if r['route']=='native').items()])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


,형식,실제 Native 재추출 elements,과거 elements,수 동일,warnings,reason
0,hwp,466,466,True,"[{""code"": ""converter_stderr"", ""detail"": ""/values: {'SOLID': UnderlineStyle.SOLID, 'DASHED': UnderlineStyle.DASHED, '...",None
1,pdf,674,674,True,"[{""code"": ""page_blank_or_image"", ""page"": 22}, {""code"": ""mupdf_warnings"", ""detail"": ""closepath with no current point\...",None


,kind,원문,원본 location,격자
0,paragraph,제 안 요 청 서,"{""format"": ""hwp"", ""section"": 0, ""path"": ""s0/p3"", ""section_path"": []}",null
1,table,사 업 명 | 한국로봇산업진흥원 사업관리시스템 \n온라인평가 체계 구축\n주관기관 | 한국로봇산업진흥원,"{""format"": ""hwp"", ""section"": 0, ""path"": ""s0/p7/t0"", ""section_path"": [], ""table_ordinal"": 1}","{""rows"": 2, ""cols"": 2, ""caption"": """", ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""사 업 명""}, {..."
2,paragraph,2025년도 중이온가속기용 극저온시스템,"{""format"": ""pdf"", ""page"": 1, ""page_label"": null, ""path"": ""p1/i0"", ""bbox"": [123.4, 156.9, 471.5, 175.9], ""section_pat...",null
3,paragraph,운전 용역 과업지시서,"{""format"": ""pdf"", ""page"": 1, ""page_label"": null, ""path"": ""p1/i1"", ""bbox"": [202.5, 191.1, 392.3, 210.0], ""section_pat...",null


,과거 Native 상태,문서 수
0,실패: hwp_empty_output,1
1,성공,96
2,실패: hwp_converter_decode_error,1


### Native 파서 대조

- HWP/PDF 원본 문서를 다시 파싱하여 기존 Native 추출 결과와 비교했다.
- HWP는 466개, PDF는 674개 Elements가 추출되어 과거 결과와 개수가 일치했다.
- 과거 Native 추출 기록에서는 전체 98개 문서 중 96개가 성공하고 2개는 HWP 관련 오류가 발생했다.
- 추출된 문단과 표의 원문 및 위치 정보를 확인했다.
- 일부 경고가 발생했으므로 전체 추출 결과가 완전히 동일한지는 추가 검증이 필요하다.

## 5. 최종 표 투영 이력

원본 좌표·셀 텍스트·기존 요소가 유일하게 대응한 표만 투영. 구조 변경과 보류 사유 확인.

In [6]:
audit=read(FINAL/'preservation_audit.json')
frame([{k:v for k,v in audit.items() if k not in ['before','after','held'] and not isinstance(v,(list,dict))}])
frame([{'보류 사유':k,'건수':v} for k,v in audit.get('held',{}).items()])
# 원본 셀/이전 표/현재 표 구조를 함께 보존한 실제 mapping 기록을 관찰합니다.
mappings=[r for r in jl(FINAL/'structure_mapping.jsonl') if r.get('kind')=='final_table_mapping']
frame([{'element_id':r['stage11_element_id'],'이전 격자':clip(r.get('previous_table'),500),'원본 확인 행':r['rows'],'원본 확인 열':r['cols'],'실제 구조 변경':r['structural_change'],'원본 stream':r['original_stream'],'원본 control record':r['original_table_control_record']} for r in mappings[:3]])

,version,documents,records,elements,chunks,native_success_documents,...,new_element_ids_created,handoff_status,structural_result,retrieval_result,protected_file_count,direct_stage8_original_raw_search_id_sha_preserved
0,final_preprocessing_v1,98,21902,26023,18976,96,...,0,CONDITIONAL PASS,A original structure fidelity improved relative to Stage11,C previous development regression performance maintained; no measured gain,736,True


,보류 사유,건수
0,no_or_multiple_stage8_element_candidates,3416
1,single_cell_table,638
2,already_verified_keep_existing,1
3,multiple_original_content_candidates,10
4,original_vs_stage8_table_order_conflict,2
5,fixed_chunker_cell_body_preservation_not_proven,58


,element_id,이전 격자,원본 확인 행,원본 확인 열,실제 구조 변경,원본 stream,원본 control record
0,2b57c93ee9149cd4e51d,"{""rows"": 2, ""cols"": 2, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""사 업 명""}, {""row"": 0, ""col""...",2,2,False,BodyText/Section0,46
1,201841e344a20526f443,"{""rows"": 3, ""cols"": 5, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""담당""}, {""row"": 0, ""col"": 1...",3,5,False,BodyText/Section0,107
2,e065e79b4db262fd2ef2,"{""rows"": 2, ""cols"": 3, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""구분""}, {""row"": 0, ""col"": 1...",2,3,False,BodyText/Section0,466


### 5. 최종 표 투영 이력

- 원본 표의 행·열·셀 정보를 기준으로 최종 Elements에 표 구조가 반영된 결과를 확인했다.
- 최종 데이터는 문서 98개, Records 21,902개, Elements 26,023개, Chunks 18,976개로 확인됐다.
- Native 추출 성공 문서는 96개이며, 최종 전달 상태는 `CONDITIONAL PASS`로 표시됐다.
- 표 구조 투영 과정에서 보류된 사유와 건수를 확인했다.
  - Stage8 요소 대응 후보 부족 또는 중복: 3,416건
  - 단일 셀 표: 638건
  - 고정 청커의 셀 본문 보존 미입증: 58건
  - 기타 보류 사유: 13건
- 화면에 표시된 표 사례 3건은 원본의 행·열 정보가 확인됐으며, 실제 구조 변경 여부는 모두 `False`였다.
- 기존 Stage11 대비 원본 표 구조 충실도가 개선된 것으로 기록되어 있다.
- 검색 성능은 이전 개발 버전 대비 유지된 것으로 기록되어 있으며, 측정된 성능 향상은 확인되지 않았다.
- 최종 전달 상태가 조건부 통과이므로 보류 항목과 제한 사항을 추가로 확인해야 한다.

## 6. 요구사항 ID·메타데이터

문서별 승인 alias와 복합 ID 처리. 같은 ID의 서로 다른 내용을 임의 병합하지 않음.

In [7]:
from requirement_metadata import 포함, 허용필드
meta_changes=read(data_versions()/'version_04_requirements_20261007T015054_7e360551/요구사항metadata_변경내역.json')
frame([{'record_id':r['record_id'],'변경 필드':list(r['변경']),'전':clip({k:v['전'] for k,v in r['변경'].items()},800),'후':clip({k:v['후'] for k,v in r['변경'].items()},800)} for r in meta_changes[:3]])
assert all(set(r['변경'])<=허용필드 for r in meta_changes)
print('저장된 metadata 변경 records:',len(meta_changes))

,record_id,변경 필드,전,후
0,03d77a47a09a31bc44eb74fafe0a26fa5f0ad25a0ebac7f0a18c24275a1367cc:body:0:4869-6362,"[requirement_aliases, requirement_ids, requirement_relations, source_requirement_fields]","{""requirement_aliases"": null, ""requirement_ids"": [], ""requirement_relations"": null, ""source_requirement_fields"": null}","{""requirement_aliases"": [{""source_id"": ""QUR-01"", ""corresponding_id"": ""QUR-001"", ""name"": ""품질관리 조직 및 품질보증 방안"", ""scope""..."
1,03d77a47a09a31bc44eb74fafe0a26fa5f0ad25a0ebac7f0a18c24275a1367cc:body:0:6445-7144,"[requirement_ids, source_requirement_fields]","{""requirement_ids"": [], ""source_requirement_fields"": null}","{""requirement_ids"": [""SFR-상담-001""], ""source_requirement_fields"": [{""source_role"": ""상세"", ""evidence_file"": ""[개인 경로 비공개]"
2,03d77a47a09a31bc44eb74fafe0a26fa5f0ad25a0ebac7f0a18c24275a1367cc:body:0:7145-8071,"[requirement_ids, source_requirement_fields]","{""requirement_ids"": [], ""source_requirement_fields"": null}","{""requirement_ids"": [""SFR-상담-002""], ""source_requirement_fields"": [{""source_role"": ""상세"", ""evidence_file"": ""[개인 경로 비공개]"


저장된 metadata 변경 records: 337


In [8]:
version=data_versions()/'version_04_requirements_20261007T015054_7e360551'
rows={r['record_id']:r for r in jl(version/'rag_input.jsonl')}
examples=[]
for change in meta_changes[:3]:
    r=rows[change['record_id']]
    assert all(r.get(k)==v['후'] for k,v in change['변경'].items())
    examples.append({'record_id':r['record_id'],'소유 요구사항':r['requirement_ids'],'본문 참조':r.get('referenced_requirement_ids'),'ID 본문 포함':{identity:포함(r['raw_text'],identity) for identity in r['requirement_ids']},'실제 본문':clip(r['raw_text']),'원본 필드 근거':clip(r.get('source_requirement_fields'),600)})
frame(examples)
frame([{'설정 문서':doc,'승인된 대응':pairs} for doc,pairs in read(LEGACY/'config/요구사항metadata_승인대응.json')['한정문서대응'].items()][:5])

,record_id,소유 요구사항,본문 참조,ID 본문 포함,실제 본문,원본 필드 근거
0,03d77a47a09a31bc44eb74fafe0a26fa5f0ad25a0ebac7f0a18c24275a1367cc:body:0:4869-6362,"[SFR-공통-001, SFR-공통-002, SFR-공통-003, SFR-상담-001, SFR-상담-002, SFR-상담-003, SFR-상담-004, SFR-상담-005, SFR-연계-001, SFR-연계-...","[PER-001, PER-002, PER-003, PER-004, PER-005, SIR-001, DAR-001, DAR-002, DAR-003, DAR-004, DAR-005, DAR-006, DAR-007...","{'SFR-공통-001': True, 'SFR-공통-002': True, 'SFR-공통-003': True, 'SFR-상담-001': True, 'SFR-상담-002': True, 'SFR-상담-003': T...",| 요구사항 구분 | 고유번호 | 요구사항 명칭 |\n| --- | --- | --- |\n| 기능 요구사항 (SFR) | SFR-상담-001 | 상담관리 (직접구매) |\n| SFR-상담-002 | 상담설정...,"[{""source_role"": ""상세"", ""evidence_file"": ""${RFP_EXTERNAL_ROOT}/user/Documents/New project/rfp_reproducible/artifacts/preproce..."
1,03d77a47a09a31bc44eb74fafe0a26fa5f0ad25a0ebac7f0a18c24275a1367cc:body:0:6445-7144,[SFR-상담-001],[],{'SFR-상담-001': True},| 요구사항 분류 | 기능 요구사항 |\n| --- | --- |\n| 요구사항 고유번호 | SFR-상담-001 |\n| 요구사항 명칭 | 상담관리 (직접구매) |\n| 요구사항 상세내용 | －상담이력관리 ◦...,"[{""source_role"": ""상세"", ""evidence_file"": ""${RFP_EXTERNAL_ROOT}/user/Documents/New project/rfp_reproducible/artifacts/preproce..."
2,03d77a47a09a31bc44eb74fafe0a26fa5f0ad25a0ebac7f0a18c24275a1367cc:body:0:7145-8071,[SFR-상담-002],[],{'SFR-상담-002': True},| 요구사항 분류 | 기능 요구사항 |\n| --- | --- |\n| 요구사항 고유번호 | SFR-상담-002 |\n| 요구사항 명칭 | 상담설정관리 (직접구매) |\n| 요구사항 상세내용 | －휴일관리 ◦...,"[{""source_role"": ""상세"", ""evidence_file"": ""${RFP_EXTERNAL_ROOT}/user/Documents/New project/rfp_reproducible/artifacts/preproce..."


,설정 문서,승인된 대응
0,4a2337a0,"{'DAR-012': 'DAR-011', 'DAR-011': 'DAR-010'}"
1,cc5d401e,"{'PMR-004': 'PMR-005', 'PMR-018': 'PMR-019'}"
2,d5812573,"{'BFR-068': 'BFR-68', 'SER-004': 'SER-04', 'SER-005': 'SER-05'}"
3,fe07779f,"{'PSR-003': 'PSR-004', 'PER–002': 'PER-002'}"


In [9]:
frame([{'변경 필드':k,'records':v} for k,v in Counter(k for r in meta_changes for k in r['변경']).items()])
expected_files=['수정_metadata보완_대상.json','새30문서_최종판정.json','30문서_필드대조.json']
frame([{'필요 입력':name,'현재 경로':str(PROJECT/'artifacts/preprocessing/요구사항42_최종검증_20261007'/name),'존재':(PROJECT/'artifacts/preprocessing/요구사항42_최종검증_20261007'/name).exists()} for name in expected_files])

,변경 필드,records
0,requirement_aliases,56
1,requirement_ids,263
2,requirement_relations,88
3,source_requirement_fields,337
4,requirement_entities,13
5,child_requirement_ids,22
6,parent_requirement_id,22


,필요 입력,현재 경로,존재
0,수정_metadata보완_대상.json,${RFP_EXTERNAL_ROOT}/project/artifacts/preprocessing/요구사항42_최종검증_20261007/수정_metadata보완_대상.json,False
1,새30문서_최종판정.json,${RFP_EXTERNAL_ROOT}/project/artifacts/preprocessing/요구사항42_최종검증_20261007/새30문서_최종판정.json,False
2,30문서_필드대조.json,${RFP_EXTERNAL_ROOT}/project/artifacts/preprocessing/요구사항42_최종검증_20261007/30문서_필드대조.json,False


### 6. 요구사항 ID·메타데이터

- 문서별 승인된 alias와 복합 요구사항 ID의 메타데이터 변경 내역을 확인했다.
- `requirement_ids`, `requirement_aliases`, `requirement_relations`, `source_requirement_fields` 등의 변경 사항을 비교했다.
- 저장된 메타데이터 변경 기록은 총 337건으로 확인됐다.
- 변경된 필드가 허용된 필드 목록에 포함되는지 검사했으며, 오류 없이 통과했다.
- 실제 Record에서 요구사항 ID와 본문 참조 ID를 비교하고, 표시된 ID가 원문에 포함되어 있는지 확인했다.
- 승인된 요구사항 ID 대응 관계를 확인했으며, 서로 다른 요구사항을 임의로 병합하지 않도록 구분했다.

#### 메타데이터 변경 현황
- `requirement_aliases`: 56건
- `requirement_ids`: 263건
- `requirement_relations`: 88건
- `source_requirement_fields`: 337건
- `requirement_entities`: 13건
- `child_requirement_ids`: 22건
- `parent_requirement_id`: 22건

#### 추가 확인 사항
- 최종 검증에 필요한 JSON 파일 3개의 존재 여부를 확인했다.
- `수정_metadata보완_대상.json`, `새30문서_최종판정.json`, `30문서_필드대조.json` 모두 현재 경로에서 `False`로 나타났다.
- 메타데이터 변경 내역은 확인했지만, 필수 검증 파일이 없어 해당 검증까지 완료된 상태는 아니다.

## 7. 정제 버전 변화

초기 재실행과 승인된 기존 버전은 서로 다른 입력. 개수만으로 동일 결과를 판단하지 않음.

In [10]:
versions=[
('이번 최초 재실행',Path(state()['initial'])/'rag_input.jsonl'),
('이번 문자 복구',Path(state()['repaired'])/'rag_input.jsonl'),
('10/7 frozen',PROJECT/'최종공유용/data/rag_input_final.jsonl'),
('Stage8_2N',HISTORY/'outputs/stage8_2r_final_v1/data/rag_input_stage8_2n_experiment.jsonl'),
('Stage10_2',HISTORY/'outputs/stage10_2_structural_candidate_v1/improved_records.jsonl'),
('Stage11 hybrid',HISTORY/'outputs/stage11_hybrid_dataset/hybrid_records.jsonl'),
('최종',FINAL/'final_records.jsonl')]
summary=[]
for name,path in versions:
    count=0;docs=set();fields=Counter()
    for r in jl(path):
        count+=1;docs.add(r.get('sha256',r.get('doc_id')));fields.update(r.keys())
    summary.append({'단계':name,'records':count,'문서':len(docs),'metadata 필드':len(fields),'파일':str(path)})
frame(summary)
print('서로 다른 버전의 record 수가 같다고 같은 데이터라고 판정하지 않습니다.')

,단계,records,문서,metadata 필드,파일
0,이번 최초 재실행,22435,98,31,${RFP_EXTERNAL_ROOT}/history/2026-10-12/notebook-9-notebook-2-1-01-2/work/preprocessing_runs/01_raw_clean_77b31...
1,이번 문자 복구,22435,98,35,${RFP_EXTERNAL_ROOT}/history/2026-10-12/notebook-9-notebook-2-1-01-2/work/preprocessing_runs/03_repaired_3c11f3...
2,10/7 frozen,21701,98,72,${RFP_EXTERNAL_ROOT}/project/최종공유용/data/rag_input_final.jsonl
3,Stage8_2N,21902,98,85,${RFP_EXTERNAL_ROOT}/history/2026-10-11/rfp-rag-1-ai-rfp-rag-2/outputs/stage8_2r_final_v1/data/rag_input_stage8...
4,Stage10_2,21902,98,87,${RFP_EXTERNAL_ROOT}/history/2026-10-11/rfp-rag-1-ai-rfp-rag-2/outputs/stage10_2_structural_candidate_v1/improv...
5,Stage11 hybrid,21902,98,88,${RFP_EXTERNAL_ROOT}/history/2026-10-11/rfp-rag-1-ai-rfp-rag-2/outputs/stage11_hybrid_dataset/hybrid_records.jsonl
6,최종,21902,98,89,${RFP_EXTERNAL_ROOT}/archives/RFP_Preprocessing_Final_v1/final_records.jsonl


서로 다른 버전의 record 수가 같다고 같은 데이터라고 판정하지 않습니다.


In [11]:
prior=HISTORY.parent.parent/'2026-10-10/rfp-rag-1-6-rag-input-2'
# 날짜별 폴더의 경로를 명시적으로 확인합니다.
prior=Path(os.path.expandvars('${RFP_EXTERNAL_ROOT}/history/2026-10-10/rfp-rag-1-6-rag-input-2'))
change_file=prior/'outputs/precision_audit_stage8_2N/format_correction_record_changes.jsonl'
if change_file.exists():
    fixes=list(jl(change_file))
    frame([{'record_id':r['record_id'],'문제 유형':r['type'],'전':clip(r['before_search_text'],1000),'후':clip(r['after_search_text'],1000),'원본 확인 표식':r['verified_added_markers'],'raw·metadata 보존':r['raw_and_coordinates_and_all_existing_metadata_unchanged']} for r in fixes[:3]])
    print('실제 수정 record 수:',len(fixes))
else:
    print('변경 원장 미발견:',change_file)

,record_id,문제 유형,전,후,원본 확인 표식,raw·metadata 보존
0,65c9785a157c8f554795a39912bac9b7e32d3431810d560ca0847d4ccdecc2ea:page:206:38-1351,B,"요구사항 번호 SER-001\n\n 요구사항 명칭 프로젝트 보안관리\n\n 정의 프로젝트 보안 관리를 위한 정보보안 관리지침 등을 준수\n\n 사업 정보 누출 방지를 위한 관리체계 수립\n - 사업 착수,...",요구사항 번호 SER-001\n\n 요구사항 명칭 프로젝트 보안관리\n\n 정의 프로젝트 보안 관리를 위한 정보보안 관리지침 등을 준수\n\n요구 사항 상세 설명\n세부 내용\n\n 사업 정보 누출 방지를...,,True
1,a6f6691e69197b5f7cdfbc3947fad8f6158dfde29d2a82d6d6429386ffab1999:ordered_page:18:stage8_1_final_child:001,B,요구사항번호 DAR-005\n\n 요구사항 명 데이터 구조 검증\n\n요구사항 분류 데이터 품질\n\n 정의 데이터 모델 검증\n\n ○ 데이터 모델 검증\n\n - 설계된 데이터 모델은 설계자·개발자·발주...,요구사항번호 DAR-005\n\n 요구사항 명 데이터 구조 검증\n\n요구사항 분류 데이터 품질\n\n 정의 데이터 모델 검증\n\n상세 설명\n세부 내용\n\n ○ 데이터 모델 검증\n\n - 설계된 데이...,,True
2,3c832bf48d6778590797a18328a78282662cd111d6f3b0b4b06362b4fc514a18:body:0:58982-59352,A,| 요구사항 고유번호 | QUR-002 |\n| --- | --- |\n| 요구사항 분류 | 품질 요구사항 |\n| 요구사항 명칭 | 기능 구현 정확성 |\n| 요구사항 상세설명 | 정의 | 기능 구현 정확성...,| 요구사항 고유번호 | QUR-002 |\n| --- | --- |\n| 요구사항 분류 | 품질 요구사항 |\n| 요구사항 명칭 | 기능 구현 정확성 |\n| 요구사항 상세설명 | 정의 | 기능 구현 정확성...,❍,True


실제 수정 record 수: 10


### 7. 정제 버전별 변화

- 전처리 단계별 Records 수와 문서 수, 메타데이터 필드 수를 비교했다.
- 전체 문서 수는 모든 버전에서 98개로 유지됐다.
- Records 수는 초기 22,435개에서 최종 21,902개로 변경됐다.

#### 버전별 변화
| 단계 | Records | 메타데이터 필드 |
|---|---:|---:|
| 최초 재실행 | 22,435 | 31 |
| 원본 문자 복구 | 22,435 | 35 |
| 10/7 Frozen | 21,701 | 72 |
| Stage8_2N | 21,902 | 85 |
| Stage10_2 | 21,902 | 87 |
| Stage11 Hybrid | 21,902 | 88 |
| 최종 | 21,902 | 89 |

- 최종 Records 수는 최초보다 533개 감소했으며, 메타데이터 필드는 31개에서 89개로 증가했다.
- Records 수가 동일하더라도 실제 데이터 내용까지 동일하다고 판단할 수는 없다.
- Stage8_2N의 원문 형식 수정 이력을 확인했으며, 실제 수정된 Record는 10개로 나타났다.
- 수정 사례에서 요구사항 ID 및 원문 텍스트의 변경 전후 내용을 확인했다.
- 각 버전의 수치와 수정 이력은 확인했지만, 모든 변경이 최종 데이터에 반영됐는지는 별도 검증이 필요하다.

## 8. 부모·자식 구간 보존

기존 부모 21,701개를 보존하고 요구사항 경계 보정으로 자식 201개 추가. 길이·토큰 기준 분할은 제외.
- 부모 Record: 분할하기 전 원본 내용을 보존
- 자식 Record: 요구사항별로 나눈 검색용 데이터

In [12]:
prior=Path(os.path.expandvars('${RFP_EXTERNAL_ROOT}/history/2026-10-10/rfp-rag-1-6-rag-input-2'))
stage7_ids={r['record_id'] for r in jl(prior/'outputs/precision_audit_stage7/rag_input_stage7.jsonl')}
final_rows=list(jl(FINAL/'final_records.jsonl'))
final_ids={r['record_id'] for r in final_rows}
assert len(stage7_ids)==21701 and stage7_ids<=final_ids
assert len(final_ids-stage7_ids)==201
families=[r for r in final_rows if r['record_id'] not in stage7_ids]
frame([{'기존 부모 records':len(stage7_ids),'추가 자식 records':len(families),'최종 records':len(final_rows)}])
frame([{'자식 record_id':r['record_id'],'parent_record_id':r.get('split_provenance',{}).get('parent_record_id'),'검색문 좌표':r.get('split_provenance',{}).get('search_range'),'원문 절대 좌표':r.get('split_provenance',{}).get('raw_absolute_range'),'요구사항':r['requirement_ids'],'본문':clip(r['search_text'],550),'원본 확인 근거':clip(r.get('split_provenance'),750)} for r in families[:3]])
# 부모 record를 남기되 검색 표현을 제외한 실제 archive 예를 확인합니다.
parents={r.get('split_provenance',{}).get('parent_record_id') for r in families}
frame([{'부모 record_id':r['record_id'],'원문 보존 길이':len(r['raw_text']),'현재 검색문 길이':len(r['search_text']),'분할 근거':clip(r.get('split_provenance'),650)} for r in final_rows if r['record_id'] in parents][:3])

,기존 부모 records,추가 자식 records,최종 records
0,21701,201,21902


,자식 record_id,parent_record_id,검색문 좌표,원문 절대 좌표,요구사항,본문,원본 확인 근거
0,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052:stage8_1_child:001,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052,"[0, 279]","[20259, 20538]",[ECR-006],| 요구사항 고유번호 | ECR-006 |\n| 요구사항 명칭 | 영상분석 S/W |\n| 요구사항 상세설명 | 정의 | 수확량 예측을 위한 영상분석 소프트웨어 |\n| 세부내용 | o 드론영상처리 S/W o...,"{""version"": ""stage8_1_verified_partition"", ""parent_record_id"": ""062655938c5bef513ae61a06ada1c052192202a0a939cd96446c..."
1,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052:stage8_1_child:002,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052,"[279, 327]","[20538, 20586]",[],5. 시스템 설계 요구사항(SDR : System Design Requirment)\n\n,"{""version"": ""stage8_1_verified_partition"", ""parent_record_id"": ""062655938c5bef513ae61a06ada1c052192202a0a939cd96446c..."
2,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052:stage8_1_child:003,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052,"[327, 619]","[20586, 20878]",[SDR-001],| 요구사항 분류 | 설계 |\n| --- | --- |\n| 고유번호 | SDR-001 |\n| 요구사항 명칭 | 운영체계 |\n| 요구사항 상세설명 | 정의 | 어플리케이션의 특성을 고려하여 운영체계(OS...,"{""version"": ""stage8_1_verified_partition"", ""parent_record_id"": ""062655938c5bef513ae61a06ada1c052192202a0a939cd96446c..."


,부모 record_id,원문 보존 길이,현재 검색문 길이,분할 근거
0,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:20259-22052,1793,0,"{""version"": ""stage8_1_verified_partition"", ""role"": ""archive_parent"", ""parent_record_id"": ""062655938c5bef513ae61a06ad..."
1,062655938c5bef513ae61a06ada1c052192202a0a939cd96446c8bed5f32e935:body:0:28135-28712,577,0,"{""version"": ""stage8_1_verified_partition"", ""role"": ""archive_parent"", ""parent_record_id"": ""062655938c5bef513ae61a06ad..."
2,16dfbd16166dcaf9907193b8ffa7433de35dc75ba22cd32dfcfe0ca23bb2d80e:body:0:26639-56436,29797,0,"{""version"": ""stage8_1_complete"", ""role"": ""archive_parent"", ""child_record_ids"": [""16dfbd16166dcaf9907193b8ffa7433de35..."


### 8. 요구사항 Record 분할 및 부모·자식 관계

- 하나의 Record에 여러 요구사항이 포함된 경우, 요구사항별로 분리한 결과를 확인했다.
- 기존 부모 Records 21,701개에서 자식 Records 201개가 추가되어 최종 21,902개로 구성됐다.
- 분할된 자식 Record에는 새로운 `record_id`가 부여되고, `parent_record_id`로 기존 부모 Record와 연결됐다.
- 자식 Record별 검색 표현, 원문 절대 좌표, 요구사항 ID 및 본문 내용을 확인했다.
- 분할된 자식의 검색·원문 범위를 구분하여 각각의 요구사항을 독립적으로 검색할 수 있도록 구성했다.
- 부모 Record는 원문 보존용으로 유지하며, 화면에 표시된 사례에서는 현재 검색문 길이가 0으로 확인됐다.
- 분할 근거와 부모·자식 연결 정보를 확인했으며, 전체 Record의 관계 검증 완료 여부는 별도로 확인이 필요하다.

## 9. Elements 보정 전후

Stage11 baseline에 적용한 실제 Patches와 최종 구조 비교.

In [13]:
REPATCH = True
if REPATCH:
    element_output=unique_output('07_final_elements.jsonl')
    command(SETTINGS['legacy_python'],FINAL/'adapter/reproduce_elements.py',['--output',element_output],'07_patches')
    save_state({'elements':element_output})
frame([checks_equal_jsonl(state()['elements'],FINAL/'final_elements.jsonl')])
patches=list(jl(FINAL/'adapter/patches.jsonl'))
frame([{'element_id':p['element_id'],'전 kind':p['before']['kind'],'후 kind':[e['kind'] for e in p['after']],'전 table':clip(p['before'].get('table'),700),'후 table':clip([e.get('table') for e in p['after']],700),'텍스트 유지':len(p['after'])==1 and p['before']['raw_text']==p['after'][0]['raw_text']} for p in patches[:3]])
print('확인된 최종 patch 수:',len(patches))

PASS 26023 c39cc6bba2fafe0e6e5720fea43b0416ee7a8017828301133e856ef3c532eaa7



,행 수,전체 값 동일,바이트 해시 동일
0,26023,True,True


,element_id,전 kind,후 kind,전 table,후 table,텍스트 유지
0,adcb7a80d3bd91cdcb8a,table,[table],"{""rows"": 5, ""cols"": 2, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""구 분""}, {""row"": 0, ""col"": ...","[{""rows"": 5, ""cols"": 2, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""구 분"", ""original_record_r...",True
1,f4d1da81bc8145f45fd2,table,[table],"{""rows"": 8, ""cols"": 17, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""구 분""}, {""row"": 0, ""col"":...","[{""rows"": 8, ""cols"": 17, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 2, ""colspan"": 1, ""text"": ""구 분"", ""original_record_...",True
2,55ba3c3da641fef99886,table,[table],"{""rows"": 3, ""cols"": 2, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""구분""}, {""row"": 0, ""col"": 1...","[{""rows"": 3, ""cols"": 2, ""cells"": [{""row"": 0, ""col"": 0, ""rowspan"": 1, ""colspan"": 1, ""text"": ""구분"", ""original_record_re...",True


확인된 최종 patch 수: 4909


### 9. Elements 보정 전후 비교

- Stage11 baseline에 최종 승인된 Patch를 적용하여 Elements의 변경 전후를 비교했다.
- 총 4,909개의 Patch 적용 기록을 확인했다.
- 표의 행·열 구조와 셀 정보가 보정된 사례를 확인했다.
- 보정 전후 모두 `kind=table`로 유지되었으며, 표 내부 구조가 변경된 사례를 확인했다.
- 화면에 표시된 사례 3건은 모두 텍스트가 유지된 것으로 나타났다.
- 최종 Elements는 26,023개로 확인됐다.
- 기존 최종 결과와 전체 값 및 바이트 SHA256 해시가 모두 일치하여 검증을 통과했다.
- 이번 검증은 저장된 Stage11 baseline에 Patch를 적용한 결과이며, 원본 HWP/PDF부터 전체 전처리 과정이 재현된 것은 아니다.

## 10. 예외·검증 기록

원본 구조·이미지 관련 미완료 검증은 남아 있음. 누락된 승인 입력은 존재 여부만 확인.

In [14]:
frame([read(Path(state()['initial']) / '전체_검증결과.json')])
frame([read(Path(state()['rechecked']) / '예외_재검사_검증.json')])
frame([{'산출물': '최종 Elements', **checks_equal_jsonl(state()['elements'], FINAL / 'final_elements.jsonl')}])

,status,source_documents,processed_representatives,failed_documents,raw_lineage_checked,rag_records,...,verified_output_supplements,source_blank_outputs,explicit_na_outputs,verified_nested_memberships,source_completeness_confirmed,note
0,검사 통과,100,98,0,22495,22435,...,5,11,1,5,False,추출·값 검출·불변조건 검사는 원본 완전성의 육안 검증을 대체하지 않음


,상태,재검사_이전예외,현재_남은예외,원문좌표_유지,정제문좌표_현재문자열_대조,본문_수치_ID_조건_무변경,원본_완전성_미확인,구조항목_임의해결,복구근거_소멸확인
0,검사 통과,6115,5529,True,True,True,True,False,588


,산출물,행 수,전체 값 동일,바이트 해시 동일
0,최종 Elements,26023,True,True


### 10. 예외·검증 기록

- 원본 추출 및 문자 복구 과정의 예외 처리와 최종 Elements 검증 결과를 확인했다.
- 원본 문서 100개 중 98개가 처리 대상으로 확인됐으며, 실패 문서는 0개로 표시됐다.
- 초기 Records는 22,435개이며, 원본 계보 확인 대상은 22,495개로 나타났다.
- 원본 추출 과정에서 출력 보완 5건, 빈 출력 11건, 명시적 N/A 1건, 중첩 관계 검증 5건을 확인했다.
- 문자 복구 예외 재검사 결과, 기존 6,115건 중 5,529건이 남았으며 복구 근거가 확인된 항목은 588건이다.
- 원문 좌표 유지, 정제문 좌표 대응, 본문·수치 ID 조건 및 원본 완전성 미확인 관련 검사는 모두 `True`로 나타났다.
- 구조 항목의 임의 해결 여부는 `False`로 확인됐다.
- 최종 Elements 26,023개는 기존 결과와 전체 값 및 바이트 해시가 모두 일치했다.
- 이번 검증은 저장된 검증 기록과 최종 결과 비교이며, 원본부터 전체 전처리 과정의 연속 재현을 의미하지는 않는다.